# Tessera — Portable Validation (Linux / Google Colab)

Bu notebook Windows mock backend testlerini GCC/Clang ile derler ve çalıştırır. **Windows native hook/unwind, gerçek MSVC ve Detours integration testlerinin yerine geçmez.**

In [ ]:
## 0. Setup private repository access

In Colab: open **Secrets** (key icon), add `GH_TOKEN`, paste your fine-grained token, and enable notebook access. Required permission: read-only Contents for `Jue-dev/Tessera`.


## 1. GCC warnings-as-errors mock test

Runner tüm backend mock testlerini derleyip çalıştırır; çıktıda `WINDOWS_MOCK_VALIDATION_COMPLETE` görünmeli.

In [ ]:
from google.colab import userdata
import os, subprocess, pathlib, time, shutil, json

TOKEN = userdata.get('GH_TOKEN')
if not TOKEN: raise RuntimeError('GH_TOKEN missing. Add it in Colab Secrets and enable notebook access.')
REPO = 'https://github.com/Jue-dev/Tessera.git'
BRANCH = 'issue-6/unwind-lifecycle'
WORKDIR = pathlib.Path('/content/tessera')
askpass.write_text(chr(10).join(['#!/bin/sh','case "$1" in *Username*) echo x-access-token ;; *) echo "$GIT_PASSWORD" ;; esac','']))
askpass.write_text('#!/bin/sh\\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GIT_PASSWORD" ;; esac\\n')
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GIT_PASSWORD':TOKEN})
try:
    clone = subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(WORKDIR)],env=env,text=True,capture_output=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GIT_PASSWORD',None)
    TOKEN = None
if clone.returncode:
    print(clone.stderr)
    raise RuntimeError(f'Clone failed ({clone.returncode}). Check token permissions, repository, and branch.')
os.chdir(WORKDIR)
HEAD = subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Clone succeeded; branch:',BRANCH,'; HEAD:',HEAD)


## 2. Clang mock test (varsa)

Clang bulunmuyorsa hücre kurar.

In [ ]:
## Test runner

Each test runs independently. Logs are saved under `/content/tessera-colab-logs/`; any failure causes the final summary to fail.


## 3. GCC AddressSanitizer + UndefinedBehaviorSanitizer

Bu testler mock harness içindeki bellek ve UB hatalarını yakalamaya yardımcı olur.

In [ ]:
LOG_DIR=pathlib.Path('/content/tessera-colab-logs'); LOG_DIR.mkdir(exist_ok=True)
results=[]

def run_test(name, command, extra_env=None):
    print(chr(10)+f'===== {name} =====')
    run_env=os.environ.copy()
    if extra_env: run_env.update(extra_env)
    start=time.time(); proc=subprocess.run(command,cwd=WORKDIR,env=run_env,text=True,capture_output=True); elapsed=round(time.time()-start,2)
    log=chr(10).join([f'$ {' '.join(command)}','','--- stdout ---',proc.stdout,'--- stderr ---',proc.stderr,f'EXIT={proc.returncode}'])
    (LOG_DIR/(name.lower().replace(' ','_')+'.log')).write_text(log)
    print(proc.stdout)
    if proc.stderr: print(proc.stderr)
    status='PASS' if proc.returncode==0 else 'FAIL'; results.append({'test':name,'status':status,'exit_code':proc.returncode,'seconds':elapsed})
    print(f'{status} (exit {proc.returncode}, {elapsed}s)'); return proc.returncode


## 1. GCC mock tests

Run the mock backend suite with GCC.

run_test('GCC mock',['bash','verify/windows_mock/run_windows_mock_tests.sh'])

## 2. Clang mock tests

Install Clang if needed, then run the same suite.

if shutil.which('clang++') is None:
    install=subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y clang'],text=True,capture_output=True)
    print(install.stdout)
    if install.returncode: print(install.stderr)
if shutil.which('clang++'): run_test('Clang mock',['bash','verify/windows_mock/run_windows_mock_tests.sh'],{'CXX':'clang++'})
else: results.append({'test':'Clang mock','status':'FAIL','exit_code':127,'seconds':None})

## 3. GCC ASan + UBSan

run_test('GCC ASan UBSan',['bash','verify/windows_mock/run_windows_mock_tests.sh'],{'EXTRA_CXXFLAGS':'-fsanitize=address,undefined -fno-omit-frame-pointer','ASAN_OPTIONS':'detect_leaks=1:halt_on_error=1','UBSAN_OPTIONS':'halt_on_error=1:print_stacktrace=1'})

## 4. Summary

print('Validated HEAD:',HEAD); print(json.dumps(results,indent=2))
(LOG_DIR/'summary.json').write_text(json.dumps({'branch':BRANCH,'head':HEAD,'results':results},indent=2))
subprocess.run(['git','status','--short'],check=False)
failures=[r for r in results if r['status']!='PASS']
if failures: raise RuntimeError(f'{len(failures)} validation(s) failed; see logs in {LOG_DIR}')
print('ALL PORTABLE VALIDATIONS PASSED')


In [ ]:
!cat verify/windows_mock/windows_mock_tests.log
!git status --short
